# Parkshare - Setup DB SQLite (sources brutes)


In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd

ROOT = Path.cwd().resolve().parent
SOURCES_DIR = ROOT / "data" / "Sources"
DB_DIR = Path.cwd() / "db"
DB_DIR.mkdir(parents=True, exist_ok=True)
DB_PATH = DB_DIR / "parkshare_raw.db"
SCHEMA_PATH = Path.cwd() / "sql" / "schema_raw.sql"

print("ROOT:", ROOT)
print("SOURCES_DIR:", SOURCES_DIR)
print("DB_PATH:", DB_PATH)
print("SCHEMA_PATH:", SCHEMA_PATH)


ROOT: C:\Users\liena\Travaux\B3 - Dev\challenge48h\parkshare_challenge48h_equipe9
SOURCES_DIR: C:\Users\liena\Travaux\B3 - Dev\challenge48h\parkshare_challenge48h_equipe9\data\Sources
DB_PATH: c:\Users\liena\Travaux\B3 - Dev\challenge48h\parkshare_challenge48h_equipe9\app\db\parkshare_raw.sqlite
SCHEMA_PATH: c:\Users\liena\Travaux\B3 - Dev\challenge48h\parkshare_challenge48h_equipe9\app\sql\schema_raw.sql


In [ ]:
files_config = {
    "AAA_OPENDATA_RECHARGEABLE_2025-11-24.csv": {
        "table": "raw_aaa_opendata_rechargeable",
        "sep": ",",
        "encoding": "latin1",
    },
    "tableau-synthetique-coproff-com-2024.csv": {
        "table": "raw_tableau_synthetique_coproff",
        "sep": ";",
        "encoding": "utf-8",
    },
    "population_INSEE.csv": {
        "table": "raw_population_insee",
        "sep": ",",
        "encoding": "utf-8",
    },
    "Mapping.csv": {
        "table": "raw_mapping",
        "sep": ";",
        "encoding": "utf-8",
    },
}

files_config

## Inspection rapide (50 premieres lignes)

In [ ]:
for file_name, cfg in files_config.items():
    path = SOURCES_DIR / file_name
    print("\n" + "=" * 80)
    print(f"Fichier: {file_name}")
    df_preview = pd.read_csv(
        path,
        sep=cfg["sep"],
        encoding=cfg["encoding"],
        dtype=str,
        nrows=50,
    )
    print(f"Colonnes ({len(df_preview.columns)}):", list(df_preview.columns))
    display(df_preview.head(5))


## Creation du schema brut

In [ ]:
assert SCHEMA_PATH.exists(), f"Schema SQL introuvable: {SCHEMA_PATH}"

with sqlite3.connect(DB_PATH) as conn:
    schema_sql = SCHEMA_PATH.read_text(encoding="utf-8")
    conn.executescript(schema_sql)
    print("Schema cree avec succes.")


## Chargement echantillon (1000 lignes / fichier)

Aucune transformation metier. Toutes les colonnes sont chargees en texte pour conserver le brut.

In [ ]:
N_SAMPLE = 1000

with sqlite3.connect(DB_PATH) as conn:
    for file_name, cfg in files_config.items():
        path = SOURCES_DIR / file_name
        table_name = cfg["table"]

        df = pd.read_csv(
            path,
            sep=cfg["sep"],
            encoding=cfg["encoding"],
            dtype=str,
            nrows=N_SAMPLE,
        )

        # Standardisation minimale de nom de colonne pour compatibilite SQL.
        # Le contenu des valeurs reste strictement brut.
        df.columns = [
            c.strip()
            .replace(" ", "_")
            .replace("-", "_")
            .replace("%", "pct")
            .replace("é", "e")
            .replace("è", "e")
            .replace("ê", "e")
            .replace("à", "a")
            .replace("û", "u")
            .replace("ô", "o")
            .replace("ç", "c")
            for c in df.columns
        ]

        df["source_file"] = file_name

        df.to_sql(table_name, conn, if_exists="append", index=False)
        print(f"{table_name}: {len(df)} lignes chargees.")


## Verification rapide

In [ ]:
tables = [
    "raw_aaa_opendata_rechargeable",
    "raw_tableau_synthetique_coproff",
    "raw_population_insee",
    "raw_mapping",
]

with sqlite3.connect(DB_PATH) as conn:
    for t in tables:
        count = conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
        print(f"{t}: {count} lignes")
